<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/coagfunc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Coagulation - Flocculation - Floc Removal calculations***

In [ ]:
import pandas as pd
import math as mh

def coagfunc(Partable2, dailyflow, Treatment,lanes):

    ## COAGULATION PROCESS
    # Constants for coagulation and floculation
    print("")
    print("COAGULATION - FLOCCULATION - FLOC REMOVAL PROCESS")
    Turbidity = mh.ceil(Partable2['Turbidity'][2])
    print("The average maximum measured turbidity (NTU) is",Turbidity)
    TSS = mh.ceil(Partable2['Total Suspended Solids (TSS)'][2]*10)/10
    print("The average maximum measured TSS (mg/l) is",TSS)
    DOC = mh.ceil(Partable2['Dissolved Organic Carbon (DOC)'][2]*10)/10
    print("The average maximum measured DOC is",DOC)
    pH = mh.ceil(Partable2['pH'][1]*10)/10
    print("The average measured pH is",pH)
    # percentage_removal_organic_matter = float(input("Enter the required percentage removal of organic matter: "))

    # Coagulation assumptions
    watervisc=0.0010005 # dynamic viscosity of water in N*s/m2
    flowlane=dailyflow/lanes #m3/day

    ##Coagulation calculations
    if (Treatment['Treatment Steps'] == "Coagulation").any() and (Treatment['Treatment Technologies'] == "Dynamic Mixing").any():
      Coag_area,Ecoag,coagheadloss,coagTankdepth=dynamic_coag(flowlane, watervisc,lanes) #, units
    elif (Treatment['Treatment Technologies'] == "Cascade Mixing").any():
      Coag_area,Ecoag,coagheadloss,coagTankdepth=cascade_coag(flowlane, watervisc,lanes) #, units
    elif (Treatment['Treatment Technologies'] == "Inline Mixing").any():
      selected_blender,Coag_area,Ecoag,coagheadloss,coagTankdepth= select_inline_blender(flowlane, watervisc,lanes)

    ##Coagulation chemicals
    totalvolumeofcoag, chemarearequired, Echem, coagtype,amountofcoagday  = coag_chem(flowlane, lanes)# , units

    ## Energy consumption
    Ecoag=Ecoag+Echem #daily energy consumption

    ## FLOCCULATION PROCESS
    floc_area,Efloc,flocheadloss,flocTankdepth = dynamic_floc(flowlane, watervisc,lanes) #, units

    ## FLOC REMOVAL
    if (Treatment['Treatment Technologies'] == "Sludge Blanket Clarification").any():
      clar_area, Eclar, clarhead, clarunitdepth=sludgeblanket_clar(flowlane,lanes,TSS,watervisc) #, units
    elif (Treatment['Treatment Technologies'] == "Horizontal Settling").any():
      clar_area, Eclar, clarhead, clarunitdepth=horizontal_clar(flowlane,lanes,TSS,watervisc) #, units
    elif (Treatment['Treatment Technologies'] == "Titled-plate Settling").any():
      clar_area, Eclar, clarhead, clarunitdepth = lamella_clar(flowlane,lanes,TSS,watervisc)
    elif (Treatment['Treatment Technologies'] == "Dissolved Air Flotation").any():
      clar_area, Eclar,clarhead, clarunitdepth  = daf_clar(flowlane,lanes,TSS,watervisc)

    return Coag_area, Ecoag,coagheadloss,coagTankdepth, floc_area, Efloc, flocheadloss,flocTankdepth, clar_area, Eclar,clarhead,clarunitdepth, chemarearequired,coagtype,amountofcoagday


## COAGULATION
def dynamic_coag(flowlane, watervisc, lanes):
# Set the number of units per lane
    print("\n--- Coagulation process Design ---")
    while True:
        units_str = input("Number of different coagulation units per lane:")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number greater than or equal to 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")
    flowunit = flowlane / units

# Coagulation tank size
    while True:
        t=int(input("Set the retention time in minutes (min 1 / max 4 default 3 )") or 3) #retention time in minutes
        try:
            if 1 <= t <= 4:
                break
            else:
                print("Invalid input. Please enter a value between 1 and 4.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    Coagunitvolume=flowunit*0.0006944444*t # 1m3/day =0.0006944 m3/min
    # Define the shape of the tanks
    print("Define the tank shape:")
    print("1. Rectangular")
    print("2. Cylindrical")
    while True:
        coagshape = input("Enter your choice (1/2): ")
        coagshapeint=int(coagshape)
        if coagshapeint==1 or coagshapeint==2:
           break
        else:
           print("Invalid input. Please enter a valid number (1 or 2).")
    if coagshape== "1":
       while True:
             coagTankdepth=float(input("Set the tank height in meters (min 2m - max 4m - default 3m):") or 3)
             try:
                 if 2 <= coagTankdepth <= 4:
                    break
                 else:
                    print("Invalid input. Please enter a value between 2 and 4.")
             except ValueError:
                 print("Invalid input. Please enter a valid number.")

       CoagTankunit=Coagunitvolume/coagTankdepth
       ##coagdepth= # the depth is two times the width
       ##CoagTankunit=CoagTankwidth**2
       Coag_area=mh.ceil(units*lanes*CoagTankunit)
    elif coagshape== "2":
       while True:
             coagTankdepth=float(input("Set the tank height in meters (min 2m - max 4m - default 3m):") or 3)
             try:
                 if 2 <= coagTankdepth <= 4:
                    break
                 else:
                    print("Invalid input. Please enter a value between 2 and 4.")
             except ValueError:
                 print("Invalid input. Please enter a valid number.")
       CoagTankunit=Coagunitvolume/coagTankdepth
       coagTankdiameter=mh.ceil((4*CoagTankunit/mh.pi))
       print("The diameter of each cercular tank is {:.2f} m".format(coagTankdiameter))
       ##coagdepth=int(input("Define the tank height in meters:")) # the depth is two times the width
       ##CoagTankunit=CoagTankwidth**2
       Coag_area=mh.ceil(units*lanes*CoagTankunit)


    print("The coagulation unit area  is {} m2".format(CoagTankunit))
    print("Total area of coagulation tanks is {} m2".format(Coag_area))

    #Power and energy of mixers
    while True:
        G = int(input("Set the velocity gradient (s⁻¹)  of the mixer (500-1000 s⁻¹ - default 1000s⁻¹ ) )") or 1000)
        try:
            if 500 <= G <= 1000:
                break
            else:
                print("Invalid input. Please enter a number between 500 and 1000.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    n=0.8 #assuming that the motor efficiency is 80%
    Pcoagunit=mh.ceil(((G**2)*watervisc*Coagunitvolume)/(n*1000)) #Power in kW
    Pcoag=mh.ceil(Pcoagunit*units*lanes) #Power in kW
    print("The mixers power is {:.2f} kW".format(Pcoag))

    Ecoag=24*Pcoag
    print("The daily coagulation energy consuption is {:.2f} kWh.".format(Ecoag))
    # Calculate Hydraulic Losses (Head Losses)

    coagheadloss = 1 + (0.001 * Coag_area) + coagTankdepth # Head loss in meters
    print("The hydraulic head of the coagulation tank is {:.2f}m.".format(coagheadloss))

    return Coag_area,Ecoag, coagheadloss,coagTankdepth

def cascade_coag(flowlane, watervisc, lanes):
    """
    Simulates cascading coagulation in drinking water treatment plants.

    Args:
        flowlane (list): Flow rates per lane (m³/day).
        watervisc (float): Dynamic viscosity of water (Pa·s).
        lanes (int): Number of treatment lanes.

    Returns:
        tuple: (Total area of coagulation tanks (m²), Total mixer power (kW), Total head loss (m))
    """
# Set the number of units per lane
    print("\n--- Coagulation process Design ---")
    while True:
        units_str = input("Number of different coagulation units per lane:")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number greater than or equal to 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # --- Step 1: Define Number of Coagulation Stages ---
    while True:
        stages_str = input("Number of cascade stages (min 2, max 4, default 2): ") or "2"
        try:
            stages = int(stages_str)
            if 2 <= stages <= 4:
                break
            else:
                print("Invalid input. Enter a number between 2 and 4.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # --- Step 2: Flow Distribution per Stage ---
    flowunit = flowlane / units
    flow_per_stage=flowunit/stages
    print(f"Flow per coagulation stage: {flow_per_stage:.2f} m³/day")

    # --- Step 3: Retention Time & Tank Sizing ---
    while True:
        t = float(input("Retention time per stage (min 1, max 3, default 3 min): ") or "3")
        if 1 <= t <= 3:
            break
        else:
            print("Invalid input. Enter a value between 1 and 3.")

    # --- Step 4: Set the height difference  (ΔΗ) per stage ---
    while True:
        DH = float(input("Set the height of the weir per stage (ΔH) (min 0.5, max 1, default 0.7m) : ") or "0.7")
        if 0.5 <= DH <= 1:
            break
        else:
            print("Invalid input. Enter a value between 0.5 and 1.")

    # Convert retention time to minutes and calculate volume
    Coag_unit_volume = flowunit * 0.0006944444 * t  # 1 m³/day = 0.0006944 m³/min

    # --- Step 4: Tank Shape Selection (Rectangular/Cylindrical) ---
    print("\nDefine tank shape:")
    print("1. Rectangular")
    print("2. Cylindrical")
    while True:
        coag_shape = input("Enter choice (1/2): ")
        if coag_shape in ["1", "2"]:
            break
        else:
            print("Invalid input. Enter 1 or 2.")

    # --- Step 4: Tank Dimensions ---
    while True:
        tank_height = float(input("tank height (min 2m, max 4m, default 3m): ") or "3")
        if 2 <= tank_height <= 4:
            break
        else:
            print("Invalid input. Enter a value between 2 and 4.")

    if coag_shape == "1":  # Rectangular
        tank_area_per_unit = Coag_unit_volume / tank_height
        Coag_area = mh.ceil(units * lanes * tank_area_per_unit)
        print(f"Each rectangular tank area: {tank_area_per_unit:.2f} m²")
    else:  # Cylindrical
        tank_diameter = mh.ceil((4 * (Coag_unit_volume / tank_height) / mh.pi) ** 0.5)
        tank_area_per_unit = mh.pi * (tank_diameter ** 2) / 4
        Coag_area = mh.ceil(units * lanes * tank_area_per_unit)
        print(f"Each cylindrical tank diameter: {tank_diameter:.2f} m")

    print(f"Total coagulation area: {Coag_area:.2f} m²")
    Coag_width_unit=mh.sqrt(tank_area_per_unit) # Assuming L/W=3
    # --- Step 5: Hydraulic Losses (Head Loss) ---

    weir_coeff = 0.7      # Cd (discharge coefficient)
    weir_length = Coag_width_unit # (m)tank width
    K_loss = 0.8          # Empirical loss coefficient

    # Flow per stage (convert m³/day to m³/s)
    Q_stage = flow_per_stage / 86400

    # Calculate head loss per weir (simplified)
    velocity_approach = Q_stage / (weir_length * DH)
    h_weir = K_loss * (velocity_approach ** 2) / (2 * 9.81)

    # Total weir head loss for all stages
    total_weir_loss = h_weir * stages

    # Add to existing head loss
    coagheadloss = tank_height + (DH * stages) + total_weir_loss

    coagheadloss = tank_height + (DH * stages)
    print(f"Total hydraulic head loss: {coagheadloss:.2f} m")

    # --- Step 6: Mixing Velocity Calculation ---

    G=mh.sqrt((1000*9.81*DH*stages)/(watervisc*t*stages*60))
    print("The velocity gradient in the cascade coagulation is {:.2f} (s⁻¹)".format(G))

    Ecoag=0
    coagTankdepth=tank_height
    return Coag_area,Ecoag, coagheadloss, coagTankdepth

def select_inline_blender(flowlane, watervisc, lanes):
    """
    Selects an in-line blender for coagulation based on flow rate and calculates hydraulic losses.

    Args:
        flowlane (list): Flow rates per lane (m³/day).
        watervisc (float): Dynamic viscosity of water (Pa·s).
        lanes (int): Number of treatment lanes.

    Returns:
        tuple: (Selected blender DataFrame, Total area (m²), Total power (kW), Head loss (m))
    """
    print("\n--- Coagulation process Design ---")

    # --- Step 1: Validate Inputs ---
    if not isinstance(flowlane, (list, float, int)):
        raise ValueError("flowlane must be a list or numeric value.")
    if lanes < 1:
        raise ValueError("Number of lanes must be ≥ 1.")

    # Convert flowlane to max flow rate (m³/day)
    Q_max = max(flowlane) if isinstance(flowlane, list) else float(flowlane)

     # --- Step 2: Retention Time & Tank Sizing ---
    while True:
        t = float(input("Retention time per stage (min 1, max 5, default 3 min): ") or "3")
        if 1 <= t <= 5:
            break
        else:
            print("Invalid input. Enter a value between 1 and 5.")
    V = Q_max * t / 86400  # Volume in m³ (Q_max converted to m³/sec)

    # --- Step 3: Blender Selection Database ---
    blender_data = {
        'Motor Power (kW)': [0.35, 0.55, 0.75, 1, 1.5, 2.25, 2.25, 2.25, 3.7, 7.5, 15, 22],
        'Length (m)': [0.3, 0.35, 0.4, 0.5, 0.6, 0.65, 0.7, 0.8, 0.85, 0.9, 1.2, 1.25],
        'Diameter (mm)': [110, 170, 220, 270, 360, 410, 460, 510, 510, 610, 710, 910],
        'Volume Range (m³)': [
            (0.001, 0.03), (0.03, 0.08), (0.08, 0.15), (0.15, 0.29),
            (0.29, 0.61), (0.61, 0.86), (0.86, 1.16), (1.16, 1.63),
            (1.63, 1.74), (1.74, 2.63), (2.63, 4.75), (4.75, 8.13)
        ]
    }
    df = pd.DataFrame(blender_data)

    # --- Step 4: Select Blender Based on Volume ---
    selected_blender = df[
        (df['Volume Range (m³)'].apply(lambda x: x[0] <= V)) &
        (df['Volume Range (m³)'].apply(lambda x: x[1] >= V))
    ].iloc[0] if not df[
        (df['Volume Range (m³)'].apply(lambda x: x[0] <= V)) &
        (df['Volume Range (m³)'].apply(lambda x: x[1] >= V))
    ].empty else None

    if selected_blender is None:
        print(f"No blender found for volume {V:.3f} m³. Consider custom design.")
        return None, None, None, None

    # --- Step 5: Calculate Blender Dimensions ---
    L = selected_blender['Length (m)']
    D_mm = selected_blender['Diameter (mm)']
    D = D_mm / 1000  # Convert mm to meters
    P_coag = selected_blender['Motor Power (kW)']
    Ecoag=24*P_coag
    print("The daily coagulation energy consuption is{:.2f} kWh".format(Ecoag))
    area_per_blender = L * D * 1.3  # 1.3 = Safety factor for spacing
    Coag_area = mh.ceil(lanes * area_per_blender)

    # --- Step 6: Hydraulic Loss Calculation ---
    rho = 1000  # Water density (kg/m³)
    g = 9.81    # Gravity (m/s²)
    Q_m3s = Q_max / 86400  # Flow rate (m³/sec)
    roughness = 0.0015e-3
    mixer_K = 4
    v = Q_m3s / (mh.pi * (D/2)**2)
    Re = (rho * v * D) / watervisc  # Reynolds number
    f = 0.316 / (Re ** 0.25) if Re < 2e4 else 0.184 / (Re ** 0.2)  # Blasius/Swamee-Jain
    h_f = f * (L*2.1 / D) * (v**2 / (2 * g))
    h_m = mixer_K * (v**2 / (2 * g))
    h_mix = 1
    coagheadloss = h_f + h_m + h_mix
    coagTankdepth = 0
    # --- Step 7: Print Results ---
    print("\n--- Selected In-Line Blender ---")
    print(f"Flow rate per lane: {Q_max/lanes:.2f} m³/day")
    print(f"Blender length: {L:.2f} m")
    print(f"Blender diameter: {D_mm} mm")
    print(f"Motor power: {P_coag:.2f} kW")
    print(f"Total area for {lanes} lanes: {Coag_area:.2f} m²")
    print(f"Hydraulic head losses: {coagheadloss:.2f} m")

    return selected_blender, Coag_area, Ecoag, coagheadloss, coagTankdepth

def coag_chem(flowlane, lanes): #, units

    print("\n--- Chemicals storage area design ---")
    print("Select coagulant:")
    print("1. Aluminium coagulant  - Aluminium Suplhate Al₂(SO₄)₃)")
    print("2. Iron coagulant - Ferric Chloride FeCl₃")
    while True:
        coagchem = input("Enter your choice (1/2): ")
        coagchemint=int(coagchem)
        if coagchemint==1 or coagchemint==2:
           break
        else:
           print("Invalid input. Please enter a valid number (1 or 2).")
    if coagchem=="1":
       coagdosing=int(input("Set the optimum dosing of alum coagulant (mg/l):")) # dosing of coaggulant
       coagdensity=1370 #1450 kg/m3 found in the litterature
       ww=0.48 #48% w/w solution
       coagtype="Alum"
    elif coagchem== "2":
       coagdosing=int(input("Set the optimum dosing of iron coagulant (mg/l):")) # dosing of coaggulant
       coagdensity=1450 # kg/m3 found in the litterature
       ww=0.40 # 40% w/w solution
       coagtype="Iron"
#    CoagTankwidth=int(input("Set the width of the coagulant tank (m):")) # dosing of coaggulant
#    CoagTankunit=CoagTankwidth**2
#    Coag_area=mh.ceil(units*lanes*CoagTankunit)
    amountofcoagday=round(lanes*coagdosing*1000*flowlane/1000000,2) #kg/day
    print("The total amount of coagulant required per day is {:.2f} kg".format(amountofcoagday))
    volumeofcoag=amountofcoagday/(coagdensity * ww) #m3/day
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1day and default is 30days): ") or "30")
        if 1 <= ndays <= 365:
            break
        else:
            print("Invalid input. Enter a value between 1 and 365.")
    totalvolumeofcoag=volumeofcoag*ndays #m3/month
    totalvolumeofcoag=mh.ceil(totalvolumeofcoag)
    chemarearequired=totalvolumeofcoag/2.5 #assuming 2.5m height
    chemarearequired=mh.ceil(chemarearequired)
    dosagepumpflow=volumeofcoag*1000/24 # m3/h to l/h
    Pchem = coagdensity * 9.81 * volumeofcoag * 5/ (86400*0.8*1000) # Kw
    Echem = 24*Pchem #KWh per day
    print("The total volume of coagulant for the storing period is {:.2f} m3".format(totalvolumeofcoag))
    print("The total storage area required is {:.2f} m2".format(chemarearequired))
    return totalvolumeofcoag, chemarearequired, Echem, coagtype,amountofcoagday


## FLOCCULATION
def dynamic_floc(flowlane, watervisc, lanes):
    # Ask for number of flocculation units per lane
    print("\n--- Flocculation Process Design ---")
    while True:
        units_str = input("Number of different flocculation units per lane: ")
        try:
            units = int(units_str)
            if units >= 1:
                break
            else:
                print("Invalid input. Please enter a number >= 1.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Calculate flow per unit
    flowunit = flowlane / units

    # Ask for number of compartments per flocculation unit
    while True:
        try:
            compartments = int(input("Number of compartments per flocculation unit (2 or 3): "))
            if compartments in [2, 3]:
                break
            else:
                print("Please enter 2 or 3.")
        except ValueError:
            print("Invalid input. Please enter a valid number.")

    # Retention time per compartment
    retention_times = []
    if compartments == 2:
        while True:
            try:
                t1 = int(input("Retention time for compartment 1 (5-10 minutes): "))
                if 5 <= t1 <= 10:
                    break
                else:
                    print("Please enter a value between 5 and 10.")
            except ValueError:
                print("Invalid input.")
        while True:
            try:
                t2 = int(input("Retention time for compartment 2 (20-30 minutes): "))
                if 20 <= t2 <= 30:
                    break
                else:
                    print("Please enter a value between 20 and 30.")
            except ValueError:
                print("Invalid input.")
        retention_times = [t1, t2]
    else:
        while True:
            try:
                t1 = int(input("Retention time for compartment 1 (5-10 minutes): "))
                if 5 <= t1 <= 10:
                    break
                else:
                    print("Please enter a value between 5 and 10.")
            except ValueError:
                print("Invalid input.")
        while True:
            try:
                t2 = int(input("Retention time for compartment 2 (10-15 minutes): "))
                if 10 <= t2 <= 15:
                    break
                else:
                    print("Please enter a value between 10 and 15.")
            except ValueError:
                print("Invalid input.")
        while True:
            try:
                t3 = int(input("Retention time for compartment 3 (10-20 minutes): "))
                if 10 <= t3 <= 20:
                    break
                else:
                    print("Please enter a value between 10 and 20.")
            except ValueError:
                print("Invalid input.")
        retention_times = [t1, t2, t3]

    total_retention = sum(retention_times)

    # Tank volume per unit
    flocunitvolume = flowunit * 0.0006944444 * total_retention  # m3

    # Tank dimensions
    flocTankdepth = 3.0  # meters
    flocTankunit = flocunitvolume / flocTankdepth
    flocTankwidth = mh.ceil((flocTankunit / 3) ** 0.5)
    floc_area = mh.ceil(units * lanes * flocTankunit)

    print("Total retention time: {} minutes".format(total_retention))
    print("Flocculation unit area: {:.2f} m2".format(flocTankunit))
    print("Total area of flocculation tanks: {:.2f} m2".format(floc_area))

    # Ask for G values per compartment
    G_values = []
    G_ranges = [(50, 80), (30, 50), (10, 30)]
    for i in range(compartments):
        while True:
            try:
                G = int(input(f"G value for compartment {i + 1} ({G_ranges[i][0]}–{G_ranges[i][1]} s⁻¹): "))
                if G_ranges[i][0] <= G <= G_ranges[i][1]:
                    G_values.append(G)
                    break
                else:
                    print("Please enter a valid G value within the range.")
            except ValueError:
                print("Invalid input.")

    # Power calculation per compartment
    n = 0.8  # motor efficiency
    total_power = 0
    print("\n--- Mixing Power per Compartment ---")
    for i in range(compartments):
        V = flocunitvolume * (retention_times[i] / total_retention)  # compartment volume
        G = G_values[i]
        power = ((G ** 2) * watervisc * V * units * lanes) / (n * 1000)  # in kW
        total_power += power
        print(f"Compartment {i + 1}: Retention = {retention_times[i]} min, G = {G}, Power = {power:.2f} kW")

    Pfloc = mh.ceil(total_power)
    print(f"\nTotal flocculation mixing power: {Pfloc:.2f} kW")

    Efloc=24*Pfloc
    print("The daily flocculation energy consuption is {:.2f} kWh".format(Efloc))

    # Headlosses
    flocheadloss = 1 + (0.001 * floc_area)
    print(f"Total hydraulic head loss: {flocheadloss:.2f} m")

    return floc_area, Pfloc, flocheadloss,flocTankdepth

## FLOC REMOVAL

def sludgeblanket_clar(flowlane, lanes, TSS,watervisc):
    """
    Designs a sludge blanket clarifier with user-defined SOR and retention time.
    Validates inputs against recommended minimum values.
    """
    print("\n--- Sludge Blanket Clarification Process Design ---")

    # --- Constants ---
    SOR_MIN = 20  # Minimum Surface Overflow Rate (m³/m²/day)
    SOR_MAX = 40  # Maximum Surface Overflow Rate
    RETENTION_MIN = 1.5  # Minimum retention time (hours)
    RETENTION_MAX = 3.0  # Maximum retention time (hours)
    SLR_MAX = 15  # Maximum Solids Loading Rate (kg/m²/day)
    DEPTH_MIN = 3.0  # Minimum tank height (m)

    # --- User Inputs with Validation ---
    while True:
        units = int(input("Number of clarification units per lane: ") or "1")
        if units >= 1:
            break
        print("Error: Must have ≥1 unit per lane.")

    # Surface Overflow Rate (SOR)
    while True:
        SOR = float(input(f"Surface Overflow Rate (SOR) [{SOR_MIN}-{SOR_MAX} m³/m²/day]: "))
        if SOR_MIN <= SOR <= SOR_MAX:
            break
        print(f"Error: SOR must be between {SOR_MIN} and {SOR_MAX}.")

    # Retention Time
    while True:
        t = float(input(f"Retention time [{RETENTION_MIN}-{RETENTION_MAX} hours]: "))
        if RETENTION_MIN <= t <= RETENTION_MAX:
            break
        print(f"Error: Retention time must be between {RETENTION_MIN} and {RETENTION_MAX} hours.")

    # --- Calculations ---
    flow_per_unit = flowlane / units  # Flow per unit (m³/day)
    clarunitarea = flow_per_unit / SOR  # Area per unit (m²)
    clarunitarea = mh.ceil(clarunitarea)  # Round up to nearest integer

    clarunitvolume = flow_per_unit * (t / 24)  # Volume (m³)
    clarunitdepth = max(DEPTH_MIN, mh.ceil(clarunitvolume / clarunitarea))  # Depth (m), min 3m

    clar_area= clarunitarea*units*lanes
    # Solids Loading Rate (SLR)
    SLRunit = (TSS * flow_per_unit) / (clarunitarea * 1000)  # kg/m²/day
    SLR= units*lanes*SLRunit

    # Hydraulic Losses (Head Loss)
    upflow_velocity = (flow_per_unit / (clarunitarea * 24 * 3600))  # m/s
    clarhead = 1.5 * (upflow_velocity ** 2) / (2 * 9.81)  # Head loss (m)

    # Sludge Production & Pump Power
    SPRlane = 0.03 * (flowlane / 24) # Sludge production (m³/h)
    SPR = lanes * SPRlane  # Total sludge production (m³/h)
    Pclar = (SPR * 1000 * 9.81 * 10) / (3600 * 0.7 * 1000)  # Pump power (kW)
    Eclar=Pclar*24 #kWh
    # --- Results ---
    print("\n=== Design Results ===")
    print(f"Area per unit: {clarunitarea:.2f} m²")
    print(f"tank height: {clarunitdepth:.2f} m")
    print(f"Total required area for clarification : {clar_area:.2f} m²")
    print(f"Solids Loading Rate (SLR): {SLR:.2f} kg/m²/day (Max allowed: {SLR_MAX})")
    print(f"Upflow velocity: {upflow_velocity * 1000:.2f} mm/s")
    print(f"Hydraulic head loss: {clarhead:.4f} m")
    print(f"Sludge production: {SPR:.2f} m³/h")
    print(f"Pump power required for all units per lane: {Pclar:.2f} kW")
    print(f"Energy consumption of the sladge blanket clarification: {Eclar:.2f} kWh")

    # --- Safety Checks ---
    if SLR > SLR_MAX:
        print(f"\n⚠️ Critical Error: SLR = {SLR:.2f} > {SLR_MAX} kg/m²/day! Increase tank area or reduce SOR.")
    if upflow_velocity > 0.001:  # 1 mm/s
        print("⚠️ Warning: Upflow velocity > 1 mm/s may wash out sludge!")

    return clar_area, Eclar, clarhead, clarunitdepth

def horizontal_clar(flowlane, lanes, TSS,watervisc):
    """
    Designs a horizontal flow clarifier with user-defined parameters.
    Validates SOR and retention time against recommended values.
    """
    # --- Constants ---
    SOR_MIN = 1.5  # Minimum Surface Overflow Rate (m³/m²/hour)
    SOR_MAX = 3.0  # Maximum Surface Overflow Rate
    RETENTION_MIN = 2.0  # Minimum retention time (hours)
    RETENTION_MAX = 6.0  # Maximum retention time (hours)
    DEPTH_MIN = 2.5  # Minimum tank height (m)
    DEPTH_MAX = 5.0  # Maximum tank height (m)
    SLR_MAX = 10.0  # Maximum Solids Loading Rate (kg/m²/day)
    SCRAPER_POWER_PER_AREA = 0.01 # kW/m² (empirical for sludge scrapers)

    print("\n--- Horizontal Clarification Process Design ---")
    # --- User Inputs with Validation ---
    while True:
        units = int(input("Number of clarification units per lane: ") or "1")
        if units >= 1:
            break
        print("Error: Must have ≥1 unit per lane.")

    # Surface Overflow Rate (SOR)
    while True:
        SOR = float(input(f"Surface Overflow Rate (SOR) [{SOR_MIN}-{SOR_MAX} m³/m²/hour]: ") or "2.0")
        if SOR_MIN <= SOR <= SOR_MAX:
            break
        print(f"Error: SOR must be between {SOR_MIN} and {SOR_MAX}.")

    # Retention Time
    while True:
        t = float(input(f"Retention time [{RETENTION_MIN}-{RETENTION_MAX} hours]: ") or "4.0")
        if RETENTION_MIN <= t <= RETENTION_MAX:
            break
        print(f"Error: Retention time must be between {RETENTION_MIN} and {RETENTION_MAX} hours.")

    # --- Calculations ---
    flow_per_unit = flowlane / units  # Flow per unit (m³/day)
    clarunitarea = (flow_per_unit / 24) / SOR  # Area per unit (m²) [Q(m³/day)/24 → m³/hour, then ÷ SOR]
    clarunitarea = mh.ceil(clarunitarea)  # Round up

    clarunitvolume = (flow_per_unit / 24) * t  # Volume (m³) [Q(m³/hour) × t(hours)]
    clarunitdepth = clarunitvolume / clarunitarea  # Depth (m)
    clar_area = clarunitarea * units * lanes
    # Enforce depth limits
    if clarunitdepth < DEPTH_MIN:
        print(f"Warning: Depth {clarunitdepth:.2f}m < {DEPTH_MIN}m. Adjusting to minimum depth.")
        clarunitdepth = DEPTH_MIN
    elif clarunitdepth > DEPTH_MAX:
        print(f"Warning: Depth {clarunitdepth:.2f}m > {DEPTH_MAX}m. Consider increasing SOR or units.")

    # Solids Loading Rate (SLR)
    SLR = (TSS * flow_per_unit) / (clarunitarea * 1000)  # kg/m²/day

    # Hydraulic Losses (Head Loss) - Empirical for horizontal flow
    velocity = (flow_per_unit / 24) / (clarunitarea * 3600)  # Velocity (m/s)
    clarhead = 0.5 + 0.05 * (velocity ** 2) / (2 * 9.81)  # Minor losses (m)

    # Sludge Scraper Power
    scraper_power_unit = clarunitarea * SCRAPER_POWER_PER_AREA  # kW per unit
    scraper_power = units * scraper_power_unit  # kW
    Pclar=scraper_power

    Eclar=Pclar*24 #kWh
    Eclar=mh.ceil(Eclar)

    # --- Results ---
    print("\n=== Horizontal Clarifier Design ===")
    print(f"Area per unit: {clarunitarea:.2f} m²")
    print(f"tank height: {clarunitdepth:.2f} m (Recommended: {DEPTH_MIN}-{DEPTH_MAX}m)")
    print(f"Total required area for clarification : {clar_area:.2f} m²")
    print(f"Flow velocity: {velocity * 1000:.2f} mm/s (Typical: 0.5-2 mm/s)")
    print(f"Solids Loading Rate (SLR): {SLR:.2f} kg/m²/day (Max recommended: {SLR_MAX})")
    print(f"Head loss: {clarhead:.4f} m ")
    print(f"Sludge Scraper Power: {Pclar:.4f} kW ")
    print(f"Energy Consumption of the horizontal clarification: {Eclar:.4f} kWh ")

    # --- Safety Checks ---
    if SLR > SLR_MAX:
        print(f"⚠️ Critical: SLR = {SLR:.2f} > {SLR_MAX} kg/m²/day! Increase area or reduce loading.")
    if velocity > 0.002:  # 2 mm/s
        print("⚠️ Warning: Velocity > 2 mm/s may resuspend settled solids!")

    return clar_area, Eclar, clarhead,clarunitdepth

def lamella_clar(flowlane, lanes, TSS,watervisc):
    """
    Designs a lamella clarifier with validated inputs for SOR, plate angle, and retention time.
    Includes hydraulic checks and solids loading rate (SLR) validation.
    """
    print("\n--- Lamella (Titled-plate Settling) Clarification Process Design ---")

    # --- Constants ---
    SOR_MIN = 20      # Minimum Surface Overflow Rate (m³/m²/day)
    SOR_MAX = 40      # Maximum Surface Overflow Rate
    ANGLE_MIN = 50    # Minimum plate angle (degrees)
    ANGLE_MAX = 70    # Maximum plate angle (degrees)
    WEIR_LOADING_MAX = 180        # Max weir loading rate (m³/m/day)
    SCRAPER_POWER_PER_AREA = 0.01 # kW/m² (empirical for sludge scrapers)
    RETENTION_MIN = 0.5  # Minimum retention time (hours)
    RETENTION_MAX = 2.0  # Maximum retention time (hours)
    SLR_MAX = 15      # Maximum Solids Loading Rate (kg/m²/day)
    PLATE_SPACING = 0.1  # Standard plate spacing (m)
    BACKWASH_FLOW_RATE = 15         # m³/m²/hour (backwash intensity)
    BACKWASH_DURATION = 10          # minutes per cycle
    BACKWASH_FREQUENCY = 2         # backwashes per day
    BACKWASH_HEAD_LOSS = 8         # meters (system head loss during backwash)
    PUMP_EFFICIENCY = 0.7         # Pump/motor efficiency

    # --- User Inputs with Validation ---
    while True:
        units = int(input("Number of units per lane [default: 1]: ") or "1")
        if units >= 1:
            break
        print("Error: Must have ≥1 unit per lane.")

    while True:
        SOR = float(input(f"Surface Overflow Rate (SOR) [{SOR_MIN}-{SOR_MAX} m³/m²/day]: ") or "30")
        if SOR_MIN <= SOR <= SOR_MAX:
            break
        print(f"Error: SOR must be between {SOR_MIN} and {SOR_MAX}.")

    while True:
        angle_deg = float(input(f"Plate angle [{ANGLE_MIN}-{ANGLE_MAX}°] (default: 60°): ") or "60")
        if ANGLE_MIN <= angle_deg <= ANGLE_MAX:
            angle_rad = mh.radians(angle_deg)
            angle_factor = 1 / mh.sin(angle_rad)  # Projection factor
            break
        print(f"Error: Angle must be between {ANGLE_MIN} and {ANGLE_MAX}°.")

    while True:
        t = float(input(f"Retention time [{RETENTION_MIN}-{RETENTION_MAX} hours] (default: 1 h): ") or "1")
        if RETENTION_MIN <= t <= RETENTION_MAX:
            break
        print(f"Error: Retention time must be between {RETENTION_MIN} and {RETENTION_MAX} hours.")

    # --- Calculations ---
    flow_per_unit = flowlane / units  # Flow per unit (m³/day)
    projected_area = (flow_per_unit / SOR) * angle_factor  # Account for plate angle
    clarunitarea = mh.ceil(projected_area)  # Rounded up (m²)

    clarunitvolume = (flow_per_unit / 24) * t  # Volume (m³) [Q (m³/h) × t (h)]
    clarunitdepth = mh.ceil(clarunitvolume / clarunitarea)  # Depth (m)
    clar_area = clarunitarea * units * lanes

    # Plate dimensions (assume 3:1 length:width ratio)
    lamellawidth = mh.ceil(mh.sqrt(clarunitarea / 3))
    lamellalength = 3 * lamellawidth
    noplates_unit = int(lamellawidth / PLATE_SPACING)  # Integer plates

    # Solids Loading Rate (SLR)
    SLRunit = (TSS * flow_per_unit) / (clarunitarea * 1000)  # kg/m²/day
    SLR=units*lanes*SLRunit
    # Hydraulic Checks
    upflow_velocity = (flow_per_unit / 24) / (clarunitarea * 3600)  # m/s
    clarhead = 1 + 0.05 * (upflow_velocity ** 2) / (2 * 9.81)  # Head loss (m)

    # Effluent Weir Loading Rate
    weir_loading = flow_per_unit / lamellalength  # m³/m/day
    if weir_loading > WEIR_LOADING_MAX:
        print(f"⚠️ Weir loading {weir_loading:.1f} > {WEIR_LOADING_MAX} m³/m/day! Increase weir length.")

    # Sludge Scraper Power
    scraper_power_unit = clarunitarea * SCRAPER_POWER_PER_AREA  # kW per unit
    scraper_power = units * scraper_power_unit  # kW
    scraper_power = mh.ceil(scraper_power)
    # --- Backwash System Design ---
    backwash_flow = clarunitarea * BACKWASH_FLOW_RATE  # m³/hour
    backwash_volume = backwash_flow * (BACKWASH_DURATION / 60)  # m³ per cycle
    daily_backwash_volume = backwash_volume * BACKWASH_FREQUENCY  # m³/day
    backwash_pump_power = (daily_backwash_volume/3600) * BACKWASH_HEAD_LOSS * 1000 * 9.81 / (1000 * PUMP_EFFICIENCY)
    backwash_pump_power = mh.ceil(backwash_pump_power)

    # Energy consumption
    scraper_energy= scraper_power*24 #kWh
    backwash_energy=backwash_pump_power*BACKWASH_DURATION*BACKWASH_FREQUENCY #kWh
    Eclar=backwash_energy + scraper_energy #kWh per day

    # --- Results ---
    print("\n=== Lamella Clarifier Design ===")
    print(f"Projected area per unit: {clarunitarea:.2f} m² (incl. {angle_deg}° angle factor)")
    print(f"Tank dimensions: {lamellalength:.2f} m (L) × {lamellawidth:.2f} m (W) × {clarunitdepth:.2f} m (D)")
    print(f"Total required area for the lamellas : {clar_area:.2f} m²")
    print(f"Number of plates per unit: {noplates_unit} (spacing = {PLATE_SPACING*100} cm)")
    print(f"Upflow velocity: {upflow_velocity*1000:.2f} mm/s (Target: 0.3–1.0 mm/s)")
    print(f"Solids Loading Rate (SLR): {SLR:.2f} kg/m²/day (Max: {SLR_MAX})")
    print(f"Head loss: {clarhead:.4f} m")
    print(f"Weir loading rate: {weir_loading:.1f} m³/m/day (Max: {WEIR_LOADING_MAX})")
    print(f"Sludge scraper power: {scraper_power:.2f} kW")
    print(f"Backwash pump power: {backwash_pump_power:.2f} kW")
    print(f"Daily Energy consumption of the lamella clarification: {Eclar:.4f} kWh")

    # --- Safety Checks ---
    if SLR > SLR_MAX:
        print(f"⚠️ Critical: SLR = {SLR:.2f} > {SLR_MAX} kg/m²/day! Increase area or reduce loading.")
    if upflow_velocity > 0.001:  # 1 mm/s
        print("⚠️ Warning: Upflow velocity > 1 mm/s may disrupt settling!")

    return clar_area, Eclar, clarhead,clarunitdepth

def daf_clar(flowlane, lanes, TSS,watervisc):

    # --- Constants ---
    SHL_MIN = 7                  # Minimum surface hydraulic loading (m³/m²/hour)
    SHL_MAX = 13                 # Maximum SHL for DAF
    AIR_RATIO = 0.05             # kg air/kg solids (typical 0.02-0.05)
    RECIRC_RATIO = 0.2                # 20% recirculation (0.1-0.3 typical)
    AIR_DENSITY = 1.204          # kg/m³ at 20°C
    WATER_DENSITY = 998          # kg/m³ at 20°C
    COMPRESSOR_EFF = 0.7         # Efficiency of air compressor
    DELTA_P = 600                # Pressure difference (kPa)
    BUBBLE_DIAMETER = 50e-6           # 50 microns (20-100 µm range)
    RETENTION_MIN = 10  # Minimum retention time (min)
    RETENTION_MAX = 60  # Maximum retention time (min)

    print("\n--- Dissolved Air Floatation Process Design ---")

    # --- Input Validation ---
    while True:
        units = int(input("Number of DAF units per lane [default: 1]: ") or 1)
        if units >= 1:
            break
        print("Error: Must have ≥1 unit per lane.")

    while True:
        SHL = float(input(f" Surface Hydraulic Loading (SHL) [{SHL_MIN}-{SHL_MAX} m³/m²/hour]: ") or 10)
        if SHL_MIN <= SHL <= SHL_MAX:
            break
        print(f"Error: SHL must be between {SHL_MIN} and {SHL_MAX}.")

    while True:
        t = float(input(f"Retention time [{RETENTION_MIN}-{RETENTION_MAX} minutes] (default: 30 minutes): ") or "30")
        if RETENTION_MIN <= t <= RETENTION_MAX:
            break
        print(f"Error: Retention time must be between {RETENTION_MIN} and {RETENTION_MAX} hours.")

    # --- DAF Tank Sizing ---
    flow_per_unit = flowlane / units
    clarunitarea = mh.ceil(flow_per_unit / (SHL * 24))  # m²
    clarunitvolume = (flow_per_unit / 24) * (t/60)         # m³
    clarunitdepth = max(2.0, mh.ceil(clarunitvolume / clarunitarea))  # Min 2m depth
    clarunitwidth=clarunitdepth/0.8 # assuming D/W=0.8
    clarunitwidth=mh.ceil(clarunitwidth)
    clarunitlength=clarunitvolume/(clarunitwidth*clarunitdepth) # length in meters
    clarunitlength=mh.ceil(clarunitlength)
    clar_area = clarunitarea * units * lanes
    clar_area = mh.ceil(1.1 * clar_area)

    # --- Air System Calculations ---
    solids_mass_unit = flow_per_unit * TSS / 1000            # kg/day (TSS in mg/L)
    air_required = AIR_RATIO * solids_mass_unit              # kg air/day
    air_requiredlane =lanes* mh.ceil(air_required)
    air_volume_lane = (air_requiredlane / AIR_DENSITY) * 0.8     # m³/day (80% saturation efficiency)
    air_volume= air_volume_lane * lanes

    # Compressor Power (kW): Q(m³/s) × ΔP(kPa) / η
    compressor_power_lane = (air_volume_lane / 86400) * DELTA_P / COMPRESSOR_EFF
    compressors_power = compressor_power_lane * lanes
    compressor_power = mh.ceil(compressor_power_lane)

    # --- Bubble Physics ---
    bubble_velocity = (9.81 * (WATER_DENSITY - AIR_DENSITY) * BUBBLE_DIAMETER**2) / (18 * watervisc)  # m/s

    # --- Recirculation System ---
    Q_recirc = RECIRC_RATIO * (flow_per_unit / 24)  # m³/hour
    recirc_pump_power = (Q_recirc * DELTA_P) / (3600 * 0.7)  # kW (η=0.7)
    pumps=units*lanes
    recirc_power=recirc_pump_power*pumps
    recirc_power=mh.ceil(recirc_power)

    # --- Saturation Tank Footprint ---
    sat_tank_vol = Q_recirc * 0.1  # 6-min retention (10% of hourly flow)
    sat_tank_area = mh.ceil(sat_tank_vol / 2)  # Assume 2m depth
    tot_sat_tank_area=sat_tank_area*units*lanes

    # --- Total required area for the DAF systems ---
    DAF_area=tot_sat_tank_area+ clar_area
    DAF_area=mh.ceil(1.05*DAF_area)

    # --- Daily energy ---
    Eclar = (recirc_power+compressor_power) * 24                 # kWh/day

    # Hydraulic Losses (Head Loss) - Empirical for horizontal flow
    velocity = (flow_per_unit / 24) / (clarunitarea * 3600)  # Velocity (m/s)
    clarhead =1 + 0.1 * (velocity ** 2) / (2 * 9.81)  # Minor losses (m)
    clarhead=mh.ceil(clarhead)

    # --- Results ---
    print("\n=== DAF Clarifier Design ===")
    print(f"Area per unit: {clarunitarea:.2f} m²")
    print(f"DAF unit tank height: {clarunitdepth:.2f} m (Min 2m for proper flotation)")
    print(f"DAF unit tank width: {clarunitwidth:.2f} m (Assuming Depth/Width=0.8)")
    print(f"DAF unit tank length: {clarunitlength:.2f} m ")
    print(f"Total DAF clarification area: {clar_area:.2f} m² ")

    print("\n=== Saturation Tank design ===")
    print("\n=== Assuming 20% recirculation flow  ===")
    print(f"Recirculation pump power (minimum 1 per lane): {recirc_pump_power:.2f} kW")
    print(f"Total recirculation power ({pumps:.2f} pumps): {recirc_power:.2f} kW")
    print(f"Recirculation area per unit: {sat_tank_area:.2f} m²")
    print(f"Total recirculation area: {tot_sat_tank_area:.2f} m²")

    print(f"\n=== Air System ===")
    print(f"Air required per lane: {air_volume_lane:.2f} m³/day")
    print(f"Total air required: {air_volume:.2f} m³/day")
    print(f"Compressor power (1 per lane): {compressor_power_lane:.2f} kW")
    print(f"Total compressor power (1 per lane): {compressor_power:.2f} kW")

    print("\n=== Bubble Performance ===")
    print(f"Bubble rise velocity: {bubble_velocity*1000:.2f} mm/s")
    print(f"Bubble residence time: {clarunitdepth / bubble_velocity:.0f} sec")

    print(f"\n=== Total area required ===")
    print("\n=== Assuming additional 5% area for the recirculation pumps and compressors===")
    print(f"Total DAF  area: {DAF_area:.2f} m² ")

    print(f"\n=== DAF Daily energy requirements ===")
    print(f"Daily Energy Consumption of the DAF treatment: {Eclar:.2f} kWh")
    print(f"\n=== DAF tanks headlosses ===")
    print(f"The headloss in the DAF tanks is: {clarhead:.2f} m")
    clar_area=DAF_area

    # --- Efficiency Checks ---
    if SHL > 10:
        print("⚠️ Warning: High SOR may reduce flotation efficiency!")
    if compressor_power > 10:
        print("💡 Consider energy recovery from compressed air!")

    return clar_area, Eclar, clarhead,clarunitdepth